# 📦 Thu thập Dữ liệu Thực tế TFT
## Teamfight Tactics Match Data Collection (Riot Games API via Kaggle)

Notebook này trình bày quy trình thu thập và chuẩn bị dữ liệu thực tế:
1. **Riot Games API**: Cơ chế thu thập dữ liệu gốc từ Riot Developer Portal
2. **Kaggle Real-world Dataset**: Sử dụng bộ dữ liệu thực tế 400.000 ván đấu từ máy chủ Hàn Quốc (KR) bao gồm 5 bậc xếp hạng (Challenger, GrandMaster, Master, Diamond, Platinum)
3. **ETL & Data Standardization**: Chuyển đổi và chuẩn hóa cấu trúc dữ liệu sang định dạng phân tích chuẩn của dự án.

In [1]:
print('Test environment: dependencies already installed; package-install cell skipped.')

Test environment: dependencies already installed; package-install cell skipped.


In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import json
import os
import requests
from collections import Counter

## 1. Riot Games API (Tham khảo)

Riot Games cung cấp API miễn phí tại [developer.riotgames.com](https://developer.riotgames.com).

**Endpoint chính:**
- `/tft/match/v1/matches/by-puuid/{puuid}/ids` - Lấy danh sách match IDs
- `/tft/match/v1/matches/{matchId}` - Chi tiết 1 trận
- `/tft/summoner/v1/summoners/by-name/{name}` - Thông tin người chơi


In [3]:
# === THAM KHẢO CƠ CHẾ RIOT API ===
# Riot Developer Portal: https://developer.riotgames.com/
# Do API cá nhân bị giới hạn tần suất (Rate Limit 20 req/1s, 100 req/2min),
# việc cào trực tiếp 50.000 trận đấu mất rất nhiều thời gian.
# Do đó, nhóm sử dụng bộ dữ liệu thực tế được cộng đồng trích xuất từ Riot API trên Kaggle.
print('✅ Cơ chế Riot API đã được khảo sát. Tiếp tục với bộ dữ liệu thực tế từ Kaggle.')

✅ Cơ chế Riot API đã được khảo sát. Tiếp tục với bộ dữ liệu thực tế từ Kaggle.


## 2. Dữ liệu Thực tế từ Kaggle (Riot Games API - Korea Server)

Bộ dữ liệu: **TFT Match Data (Challenger, GrandMaster, Master, Diamond, Platinum)**
- Nguồn: [Kaggle - TFT Match Data](https://www.kaggle.com/datasets/gyejr95/tft-match-data)
- Nền tảng: Dữ liệu trận đấu thực tế trích xuất từ Riot API khu vực Hàn Quốc (KR)
- Tổng cộng 5 file CSV tương ứng 5 bậc xếp hạng từ Platinum đến Challenger:
  1. `TFT_Challenger_MatchData.csv` (~80.000 records)
  2. `TFT_GrandMaster_MatchData.csv` (~80.000 records)
  3. `TFT_Master_MatchData.csv` (~80.000 records)
  4. `TFT_Diamond_MatchData.csv` (~80.000 records)
  5. `TFT_Platinum_MatchData.csv` (~80.000 records)

## 3. Khảo sát & Chuẩn hóa Dữ liệu Thực tế sang Định dạng Dự án

Dữ liệu được chuẩn hóa thông qua module `convert_kaggle_data.py`:
- Trích xuất danh sách tướng (`champions`) từ dict `champion`
- Trích xuất trang bị của từng tướng (`items`)
- Trích xuất số sao của tướng (`champion_stars`)
- Trích xuất các Tộc/Hệ kích hoạt (`active_traits`) từ `combination`
- Gán nhãn kết quả (`placement` từ trường `Ranked`)

In [3]:
import glob

kaggle_dir = '../data/kaggle_raw'
files = sorted(glob.glob(os.path.join(kaggle_dir, '*.csv')))
print(f'🔍 Tìm thấy {len(files)} files dữ liệu thực tế từ Kaggle:')
total_raw = 0
for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    with open(f, 'rb') as fp:
        lines = sum(1 for _ in fp) - 1
    total_raw += lines
    print(f'  - {os.path.basename(f)}: {lines:,} records ({size_mb:.2f} MB)')

print(f'\n Tổng số bản ghi thực tế: {total_raw:,} records')

🔍 Tìm thấy 5 files dữ liệu thực tế từ Kaggle:
  - TFT_Challenger_MatchData.csv: 79,999 records (39.28 MB)
  - TFT_Diamond_MatchData.csv: 80,000 records (39.70 MB)
  - TFT_GrandMaster_MatchData.csv: 80,000 records (39.54 MB)
  - TFT_Master_MatchData.csv: 79,999 records (39.17 MB)
  - TFT_Platinum_MatchData.csv: 80,000 records (38.92 MB)

 Tổng số bản ghi thực tế: 399,998 records


In [4]:
# Kiểm tra file dữ liệu chuẩn hóa đã sẵn sàng trong data/raw/
csv_path = '../data/raw/tft_matches.csv'
if os.path.exists(csv_path):
    size_mb = os.path.getsize(csv_path) / (1024 * 1024)
    print(f'✅ File dữ liệu thực tế đã chuẩn hóa: {csv_path}')
    print(f'   Kích thước: {size_mb:.2f} MB')
else:
    print('⏳ Đang tiến hành chuẩn hóa từ data/kaggle_raw...')
    from src.convert_kaggle_data import convert_kaggle_dataset
    convert_kaggle_dataset()

✅ File dữ liệu thực tế đã chuẩn hóa: ../data/raw/tft_matches.csv
   Kích thước: 210.28 MB


In [5]:
# Đọc mẫu dữ liệu thực tế đã chuẩn hóa
df = pd.read_csv(csv_path, nrows=10000)
print(f'Shape mẫu (10,000 dòng): {df.shape}')
print(f'Danh sách cột: {list(df.columns)}')
df.head(5)

Shape mẫu (10,000 dòng): (10000, 14)
Danh sách cột: ['match_id', 'player_id', 'placement', 'tier', 'level', 'rounds_survived', 'game_duration_s', 'champions', 'champion_stars', 'items', 'active_traits', 'num_champions', 'num_items', 'num_active_traits']


,match_id,player_id,placement,tier,level,rounds_survived,game_duration_s,champions,champion_stars,items,active_traits,num_champions,num_items,num_active_traits
0,KR_4247538593,KR_4247538593_1,1,Challenger,8,35,2142.470703,"[""JarvanIV"", ""Sona"", ""Rakan"", ""XinZhao"", ""Neek...","{""JarvanIV"": 3, ""Sona"": 3, ""Rakan"": 3, ""XinZha...","{""JarvanIV"": [27], ""Sona"": [46], ""Rakan"": [37,...","{""DarkStar"": 2, ""Protector"": 4, ""Rebel"": 1, ""S...",8,8,7
1,KR_4247538593,KR_4247538593_2,2,Challenger,9,35,2142.470703,"[""Malphite"", ""Yasuo"", ""Sona"", ""MasterYi"", ""Jin...","{""Malphite"": 2, ""Yasuo"": 2, ""Sona"": 2, ""Master...","{""Malphite"": [7], ""Yasuo"": [99, 22, 23], ""Kayl...","{""Blaster"": 2, ""Mercenary"": 1, ""Rebel"": 6, ""Se...",9,12,9
2,KR_4247538593,KR_4247538593_3,3,Challenger,8,34,2142.470703,"[""KaiSa"", ""Annie"", ""Shaco"", ""Rumble"", ""Karma"",...","{""KaiSa"": 2, ""Annie"": 2, ""Shaco"": 2, ""Rumble"":...","{""KaiSa"": [99, 2, 23], ""Annie"": [37], ""Shaco"":...","{""Cybernetic"": 1, ""DarkStar"": 3, ""Demolitionis...",8,10,8
3,KR_4247538593,KR_4247538593_4,4,Challenger,8,33,2142.470703,"[""KaiSa"", ""Annie"", ""Shaco"", ""Rumble"", ""Fizz"", ...","{""KaiSa"": 2, ""Annie"": 2, ""Shaco"": 2, ""Rumble"":...","{""KaiSa"": [44, 37], ""Shaco"": [15, 16, 14], ""Fi...","{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 1,...",8,10,11
4,KR_4247538593,KR_4247538593_5,5,Challenger,9,33,2142.470703,"[""Ziggs"", ""Yasuo"", ""Sona"", ""MasterYi"", ""Jinx"",...","{""Ziggs"": 1, ""Yasuo"": 1, ""Sona"": 1, ""MasterYi""...","{""MasterYi"": [5], ""MissFortune"": [12, 79], ""Ga...","{""Blaster"": 2, ""Demolitionist"": 2, ""Mercenary""...",9,9,10


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           10000 non-null  str    
 1   player_id          10000 non-null  str    
 2   placement          10000 non-null  int64  
 3   tier               10000 non-null  str    
 4   level              10000 non-null  int64  
 5   rounds_survived    10000 non-null  int64  
 6   game_duration_s    10000 non-null  float64
 7   champions          10000 non-null  str    
 8   champion_stars     10000 non-null  str    
 9   items              10000 non-null  str    
 10  active_traits      10000 non-null  str    
 11  num_champions      10000 non-null  int64  
 12  num_items          10000 non-null  int64  
 13  num_active_traits  10000 non-null  int64  
dtypes: float64(1), int64(6), str(7)
memory usage: 5.3 MB


In [7]:
# Phân phối theo bậc xếp hạng và placement
print('=== Phân phối bậc xếp hạng (Rank Tier) ===')
print(df['tier'].value_counts())

print('\n=== Phân phối thứ hạng (Placement) ===')
print(df['placement'].value_counts().sort_index())

=== Phân phối bậc xếp hạng (Rank Tier) ===
tier
Challenger    10000
Name: count, dtype: int64

=== Phân phối thứ hạng (Placement) ===
placement
1    1250
2    1250
3    1250
4    1250
5    1250
6    1250
7    1250
8    1250
Name: count, dtype: int64


In [8]:
# Xem 1 bản ghi chi tiết thực tế
sample = df.iloc[0]
print('=== Chi tiết một ván đấu thực tế (Top 1 Challenger) ===')
for col in df.columns:
    print(f'{col}: {sample[col]}')

=== Chi tiết một ván đấu thực tế (Top 1 Challenger) ===
match_id: KR_4247538593
player_id: KR_4247538593_1
placement: 1
tier: Challenger
level: 8
rounds_survived: 35
game_duration_s: 2142.470703125
champions: ["JarvanIV", "Sona", "Rakan", "XinZhao", "Neeko", "Karma", "Soraka", "Lulu"]
champion_stars: {"JarvanIV": 3, "Sona": 3, "Rakan": 3, "XinZhao": 3, "Neeko": 2, "Karma": 2, "Soraka": 2, "Lulu": 1}
items: {"JarvanIV": [27], "Sona": [46], "Rakan": [37, 69], "XinZhao": [69, 25, 25], "Lulu": [59]}
active_traits: {"DarkStar": 2, "Protector": 4, "Rebel": 1, "Set3_Celestial": 3, "Set3_Mystic": 4, "StarGuardian": 2, "TemplateTrait": 1}
num_champions: 8
num_items: 8
num_active_traits: 7
